# 07 — Arbitrages et recommandation : quel modèle pour HEC Match ?

Synthèse des quatre dimensions pour les 6 modèles, **mesurées de façon homogène** : une seule règle de décision pour
tous (l'app montre à chaque utilisateur ses **3 candidats les mieux notés**, cf. `06_economics`).

| Dimension | Source | Mesure retenue |
|---|---|---|
| Performance statistique | ce notebook | AUC test, IC 95 % par bootstrap de sessions |
| Performance économique | `06_economics` | gain de part de « oui » dans le top 3 vs hasard |
| Explicabilité | `02_interpretability` | méthode disponible et fidélité |
| Stabilité | ce notebook (protocole de Rémi, `src/stability.py`) | profils du top 3 qui changent après réentraînement |
| Fairness | ce notebook (esprit de la décomposition de Max) | chances d'être dans un top 3, par origine du candidat |

Pourquoi recalculer stabilité et fairness ? `04_stability` ne couvre que les modèles de base, et l'audit de fairness
compare les modèles à des seuils différents. Ici, tout est comparable. Code : `src/tradeoffs.py`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import numpy as np
import pandas as pd
from IPython.display import Image, display

from src.tradeoffs import *

OUT_DIR.mkdir(parents=True, exist_ok=True)
ctx = load_context()
train, test, F = ctx["train"], ctx["test"], ctx["features"]
eco = pd.read_csv(REPO_ROOT / "reports" / "economics" / "top_k_par_utilisateur.csv")
interp = json.loads((REPO_ROOT / "reports" / "interpretability" / "summary.json").read_text())
summary = {}

## 1. Performance statistique

In [ ]:
perf = pd.DataFrame({NAMES[m]: {"auc": roc_auc_score(test.dec, test[f"proba_{m}"]),
                                "ic_bas": auc_ci(test, f"proba_{m}")[0], "ic_haut": auc_ci(test, f"proba_{m}")[1]}
                     for m in MODELS}).T
summary["auc"] = perf.round(3).to_dict("index")
perf.round(3)

Les intervalles se chevauchent tous largement (6 sessions de test) : **aucun modèle n'est significativement meilleur**
(conclusion identique à `04_stability`). La performance statistique ne départage pas les modèles.

## 2. Fairness : qui a ses chances d'être montré ?

In [ ]:
fc = fairness_ci_table(test)
fc.to_csv(OUT_DIR / "fairness_exposition_top3.csv", index=False)
gender = fairness_table(test)[["Femmes / hommes"]]
display(Image(plot_fairness_forest(fc)))
piv = fc.pivot(index="strategie", columns="groupe", values="ratio")
piv["distorsion (0 = parité)"] = piv.apply(lambda r: distortion(r.values), axis=1)
piv["écarts significatifs"] = fc.assign(s=(fc.ic_haut < 1) | (fc.ic_bas > 1)).groupby("strategie").s.sum()
summary["fairness_ratios"] = piv.round(3).to_dict("index")
summary["genre_ratio_top3"] = gender.round(3).to_dict()["Femmes / hommes"]
piv.round(2)

**Lecture.** Chaque point est le rapport « chances d'être dans le top 3 d'un utilisateur » entre un groupe et les
candidats blancs (1 = parité). Référence en gris : le même rapport dans les **décisions réelles** des participants.

- **XGBoost mitigé** est le seul modèle proche de la parité pour les trois groupes, et au niveau des décisions réelles
  (distorsion ≈ 0,16 contre 0,14 dans les décisions humaines) : il n'amplifie pas les préférences.
- **Le logit** amplifie fortement les écarts : les candidats noirs et asiatiques ont environ deux fois moins de chances
  d'être montrés, **de façon significative**, alors que l'écart réel est faible. La mitigation de Max réduit l'écart
  sans le supprimer.
- **TabICL de base** sous-expose significativement les candidats noirs ; sa version mitigée est intermédiaire.
- Les intervalles sont larges (130 candidats noirs et 135 latinos seulement dans le test) : ce sont des tendances,
  à surveiller en production.

**Genre** : dans ce speed dating, chacun ne rencontre que des personnes de l'autre sexe, donc les candidats d'un
utilisateur sont tous du même genre. Un classement *par utilisateur* ne peut donc pas favoriser un genre
(rapport ≈ 0,98 pour tous les modèles, contre 1,79 dans les « oui » réels). **Le design « top k par utilisateur »
neutralise le biais de genre par construction** ; un seuil global, lui, ne le ferait pas (cf. `02_interpretability`,
section 10).

## 3. Stabilité : le modèle change-t-il d'avis si on le réentraîne ?

In [ ]:
refits = stability_refits(train, test, F, n_refits=100)
refits.to_csv(OUT_DIR / "stabilite_refits.csv", index=False)
stab = stability_summary(refits)
display(Image(plot_stability(stab)))
summary["stabilite"] = stab.round(3).to_dict("index")
stab.round(3)

**Lecture.** Protocole de Rémi (100 réentraînements sur des sessions tirées au hasard, même test). On retrouve ses
chiffres : environ 16 % de décisions qui basculent pour le logit, 21 % pour XGBoost. Traduit pour l'app :
**environ 3 profils recommandés sur 10 changent pour le logit, 4 sur 10 pour XGBoost** quand on réentraîne sur
d'autres soirées. La mitigation ne change rien à la stabilité. TabICL n'est pas mesurable (réentraînement GPU uniquement).

## 4. Tableau des arbitrages

In [ ]:
def fmt(x, d=2):
    return f"{x:.{d}f}".replace(".", ",")

top3 = eco[eco.k == 3].set_index("strategie")
card = {}
for m in MODELS:
    n = NAMES[m]
    a = perf.loc[n]
    g = top3.loc[n]
    row = {}
    row["AUC test\n(IC 95 %)"] = (f"{fmt(a.auc)}\n[{fmt(a.ic_bas)} ; {fmt(a.ic_haut)}]", "moyen")
    sig = g.ic_bas > 0
    row["Gain top 3\nvs hasard"] = (f"+{fmt(g.gain_vs_hasard_pp, 1)} pts\n[{fmt(g.ic_bas, 1)} ; {fmt(g.ic_haut, 1)}]",
                                     "bon" if sig else "moyen")
    if m.startswith("logit"):
        row["Explicabilité"] = ("Native : coefficients\n+ IC, lisible", "bon")
    elif m.startswith("xgb"):
        row["Explicabilité"] = ("SHAP exact,\naffichable dans l'app", "moyen")
    else:
        row["Explicabilité"] = (f"Surrogate seulement\n(fidélité R² {fmt(interp['tabicl_surrogate']['xgb_profondeur_2']['r2_cv_by_session'])})", "faible")
    if m in stab.index:
        v = stab.loc[m, "top3_change"]
        row["Stabilité\n(top 3 qui change)"] = (f"{v:.0%}".replace("%", " %"), "bon" if v < 0.35 else "faible")
    else:
        row["Stabilité\n(top 3 qui change)"] = ("Non mesurable\n(GPU)", "na")
    r = piv.loc[n]
    lvl = "faible" if r["écarts significatifs"] > 0 else ("bon" if r["distorsion (0 = parité)"] < 0.25 else "moyen")
    row["Fairness origine\n(asiat. · latinos · noirs)"] = (
        f"{fmt(r['Asiatiques'])} · {fmt(r['Latinos'])} · {fmt(r['Noirs'])}", lvl)
    row["Déploiement"] = (("GPU requis", "faible") if "tabicl" in m else ("CPU, léger", "bon"))
    card[n] = row
card = pd.DataFrame(card).T
display(Image(plot_scorecard(card)))
card.map(lambda c: c[0].replace("\n", " ")).to_csv(OUT_DIR / "tableau_arbitrages.csv")
card.map(lambda c: c[0].replace("\n", " "))

## 5. Recommandation

**Recommandation : XGBoost mitigé, déployé en mode « top k par utilisateur », avec trois garde-fous.**

**Pourquoi lui.** À performance statistiquement équivalente (aucun écart d'AUC significatif), c'est le seul modèle qui
**n'amplifie pas les écarts d'exposition entre origines** : il reste au niveau des décisions humaines, sans écart
significatif à la parité, là où le logit divise par deux les chances des candidats noirs et asiatiques. Il s'explique
bien (SHAP exact, affichable à l'utilisateur : « pourquoi ce profil vous est proposé »), tourne sur CPU, et le mode
top k neutralise le biais de genre.

**Ce qu'on accepte en échange.**
- **Moins de stabilité** : environ 4 profils recommandés sur 10 changent à chaque réentraînement (3 sur 10 pour le logit).
- **Moins de lisibilité que le logit** : on passe d'un modèle lisible par construction à une explication post-hoc.
- **Un gain en top 3 non significatif à lui seul** (+5,4 points, IC qui touche zéro), mais du même ordre que les autres.

**Pourquoi pas les autres.**
- **Logit mitigé** : le plus stable et le plus transparent, un vrai argument face à un régulateur. Mais il amplifie les
  écarts entre origines, ce qui est rédhibitoire pour une app qui décide qui est *vu*.
- **TabICL mitigé** : meilleure AUC et meilleur classement par utilisateur, mais c'est le moins explicable (un tiers
  seulement de ses prédictions expliqué par le meilleur surrogate), sa stabilité n'est pas mesurable, et il exige un GPU.
  Le gain d'AUC (non significatif) ne compense pas cette opacité pour une IA de confiance.

**Les garde-fous.**
1. **Stabiliser** : réentraîner à cadence fixe et moyenner plusieurs réentraînements (bagging), pour éviter que les
   recommandations d'un utilisateur changent du jour au lendemain.
2. **Surveiller la fairness en continu** : suivre les rapports d'exposition par origine (tableau de la section 2) et
   déclencher une alerte sous la règle des 4/5.
3. **Mesurer le vrai coût d'une mauvaise recommandation** (A/B test sur le désabonnement) : c'est lui qui détermine si
   le modèle rapporte beaucoup ou presque rien (`06_economics`, section 3).

**Limites** : 6 sessions de test (intervalles larges), données de 2002–2004 limitées à des étudiants de Columbia,
P&L construit sur des hypothèses, variable `imprace_A` (préférence raciale déclarée de l'utilisateur) toujours n° 1
des variables du modèle : son maintien est un choix éthique à trancher explicitement.

In [ ]:
summary["recommandation"] = "XGBoost mitigé, mode top k par utilisateur, avec stabilisation, suivi de fairness et A/B test des coûts"
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=float))
print("ok")